# Lesson 13 — Cutting Planes

## Learning objectives

1. Derive a Gomory cut from a fractional simplex tableau {cite:p}`Gomory1958`.
2. Recognize Chvátal-Gomory, MIR, lift-and-project, and split cuts.
3. Understand cut management in modern solvers (cut pool, separation, age).
4. Use `discopt` cut callbacks.

## 1. The cutting-plane idea

If the LP relaxation $z_{LP}$ is fractional, find a linear inequality $\alpha^\top x \le \beta$ that:

- holds for every integer-feasible $x$ (valid),
- is *violated* by the current LP optimum $x_{LP}$.

Add it; resolve. Repeat. Each iteration tightens the relaxation. {cite:p}`Wolsey1998,Conforti2014` are the textbook references.

## 2. Gomory mixed-integer cut

From a row of the optimal tableau where basic variable $x_i$ is fractional, with fractional part $f_i$ and elements $a_{ij}$ across non-basic columns, the Gomory cut is

$$\sum_{j \in N} \frac{f_{ij} (1 - f_i) + (1 - f_{ij}) f_i}{1 - f_i} x_j \ge f_i$$

(formula simplified; see {cite:t}`Gomory1958` or {cite:p}`Wolsey1998` for the careful derivation). It's a separation cut against $x_{LP}$ that preserves all integer feasibility.

## 3. Mixed-integer rounding (MIR)

Take any valid inequality $\sum a_j x_j \ge b$ where $x_j \in \mathbb{Z}_+$. Apply integer rounding:

$$\sum \lfloor a_j \rfloor x_j + \sum_{j: f_j > f_b} \frac{f_j - f_b}{1 - f_b} x_j \ge \lceil b \rceil$$

with $f_a, f_b$ fractional parts. MIR generalizes Gomory and is one of the most effective cut families in production solvers {cite:p}`Cornuejols2008`.

## 4. Lift-and-project

For 0/1 problems: form the disjunction $x_j = 0 \vee x_j = 1$, take the convex hull of the two LP relaxations, and project onto $x$-space. Yields a polynomial-time procedure for generating cuts {cite:p}`Balas1993`.

## 5. Cut management

Modern MILP solvers add hundreds of cuts per node. They:

- **Pool** cuts that aren't currently violated.
- **Age** them out if not used in N nodes.
- Apply cuts globally (root) vs locally (single subtree).
- Use **strong cuts** (deep, dense) sparingly and **weak cuts** (sparse, cheap) often.

Cut quality is measured by depth: distance from $x_{LP}$ to the cut hyperplane.

In [ ]:
import numpy as np, discopt as do

# Toy MILP where Gomory cuts close the gap
# max x1 + x2 s.t. 2x1 + x2 <= 5, x1 + 3x2 <= 6, x1,x2 in {0,1,2,...}
m = do.Model("cut_demo")
x = m.add_variables(2, vtype="integer", lb=0, ub=5, name="x")
m.add_constraint(2*x[0] + x[1] <= 5)
m.add_constraint(x[0] + 3*x[1] <= 6)
m.maximize(x[0] + x[1])

r = m.solve(cuts=False, log_relaxation=True)
print("no cuts: nodes =", r.nodes, " z* =", r.objective)
r = m.solve(cuts=True,  log_relaxation=True)
print("with cuts: nodes =", r.nodes, " z* =", r.objective)


## References
{cite:p}`Gomory1958` (original), {cite:p}`Balas1993` (lift-and-project),
{cite:p}`Cornuejols2008` (modern survey),
{cite:p}`Wolsey1998,Conforti2014` (textbooks).